# 📅 Day 03: Fenwick Tree (Binary Indexed Tree)

---

## Why Another Tree When We Have Segment Tree?

Fenwick tree does the same thing as segment tree for **prefix sums** — but it's way simpler to code. In an interview, if you need range sums with updates, Fenwick is often the move because you can write it in ~15 lines.

The trade-off: Fenwick only works for **prefix-based operations** (sum, count). Can't do range min/max — for that you need a full segment tree.

### The Lowbit Trick: `i & (-i)`

This is the magic behind Fenwick trees. `i & (-i)` gives the **lowest set bit** of `i`:

```
  i = 6  → binary 110 → lowest set bit = 010 = 2
  i = 12 → binary 1100 → lowest set bit = 100 = 4
  i = 7  → binary 111 → lowest set bit = 001 = 1
```

Each index `i` in the BIT stores the sum of `lowbit(i)` elements ending at index `i`:

```
  Array: [_, a1, a2, a3, a4, a5, a6, a7, a8]  (1-indexed)
  BIT:   [_, a1, a1+a2, a3, a1..a4, a5, a5+a6, a7, a1..a8]
  
  prefix_sum(6):
    tree[6] → covers a5+a6    (6 = 110, lowbit = 2)
    6 - 2 = 4
    tree[4] → covers a1..a4   (4 = 100, lowbit = 4)
    4 - 4 = 0 → stop
    Answer = tree[6] + tree[4] ✓
```

```
╔═══════════════════════════════════════════════════════════════════════╗
║  FENWICK vs SEGMENT TREE                                           ║
║                                                                     ║
║  Fenwick:  ~15 lines, O(n) space, prefix sums only               ║
║  Segment:  ~30 lines, O(2n) space, any associative op            ║
║                                                                     ║
║  Both: O(log n) query, O(log n) update                            ║
║  Fenwick wins on: simplicity and constant factor                   ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
class FenwickTree:
    """
    1-indexed Binary Indexed Tree for prefix sums.
    update: walk UP (add lowbit)
    query: walk DOWN (subtract lowbit)
    """
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)  # 1-indexed!
    
    def update(self, i, delta):
        """Add delta to index i. Walk UP by adding lowbit."""
        while i <= self.n:
            self.tree[i] += delta
            i += i & (-i)
    
    def prefix_sum(self, i):
        """Sum of [1..i]. Walk DOWN by subtracting lowbit."""
        total = 0
        while i > 0:
            total += self.tree[i]
            i -= i & (-i)
        return total
    
    def range_sum(self, left, right):
        return self.prefix_sum(right) - self.prefix_sum(left - 1)

bit = FenwickTree(5)
for i, v in enumerate([1, 3, 5, 7, 9], 1):
    bit.update(i, v)

print(bit.prefix_sum(3))    # 9  (1+3+5)
print(bit.range_sum(2, 4))  # 15 (3+5+7)

---

## 🎯 Problem: Count of Smaller Numbers After Self (LC #315)

For each element, count how many elements to its right are smaller.

### The Idea

Process **right to left**. Use a Fenwick tree indexed by **value** (not position). For each number:
1. **Query**: how many values smaller than me? → `prefix_sum(rank - 1)`
2. **Update**: mark this value as seen → `update(rank, 1)`

Since values can be huge, first do **coordinate compression** — map values to ranks 1..k.

```
  nums = [5, 2, 6, 1]
  Sorted unique: [1, 2, 5, 6] → rank: {1:1, 2:2, 5:3, 6:4}
  
  Process right to left:
    i=3: num=1, rank=1. query(0)=0. update(1). BIT has {1}
    i=2: num=6, rank=4. query(3)=1 (rank 1 seen). update(4). BIT has {1,6}
    i=1: num=2, rank=2. query(1)=1 (rank 1 seen). update(2). BIT has {1,2,6}
    i=0: num=5, rank=3. query(2)=2 (ranks 1,2 seen). update(3).
    
  Result: [2, 1, 1, 0]
```

In [ ]:
def countSmaller(nums):
    """
    Fenwick tree on coordinate-compressed values.
    Time: O(n log n), Space: O(n)
    """
    sorted_unique = sorted(set(nums))
    rank = {v: i+1 for i, v in enumerate(sorted_unique)}
    
    bit = FenwickTree(len(sorted_unique))
    result = []
    
    for num in reversed(nums):
        r = rank[num]
        result.append(bit.prefix_sum(r - 1))  # Count smaller
        bit.update(r, 1)                       # Mark seen
    
    return result[::-1]

print(countSmaller([5,2,6,1]))  # [2,1,1,0]

---

## 🗺️ Fenwick Tree Pattern Map

```
╔═══════════════════════════════════════════════════════════════════════╗
║  FENWICK TREE (BIT) PATTERNS                                         ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  CORE: Binary Indexed Tree for prefix sums with point updates       ║
║  → update(i, delta): add delta to index i — O(log n)               ║
║  → query(i): prefix sum [0..i] — O(log n)                          ║
║  → range_query(l, r): query(r) - query(l-1)                        ║
║                                                                       ║
║  BIT TRICK: i & (-i) gives lowest set bit                           ║
║  → Update: walk UP the tree (i += i & (-i))                        ║
║  → Query: walk DOWN the tree (i -= i & (-i))                       ║
║                                                                       ║
║  APPLICATIONS:                                                        ║
║  → Count inversions / count smaller after self                       ║
║  → Range sum with updates (simpler than segment tree)                ║
║  → Coordinate compression + BIT for ranking queries                  ║
║                                                                       ║
║  vs SEGMENT TREE: BIT is simpler but less flexible                   ║
║  → BIT: point update + prefix query. Segment tree: range query.     ║
║                                                                       ║
╚═══════════════════════════════════════════════════════════════════════╝
```

---

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 315 | Count Smaller After Self | Hard | BIT + coordinate compression |
| 493 | Reverse Pairs | Hard | BIT or merge sort |
| 327 | Count of Range Sum | Hard | BIT on prefix sums |
| 307 | Range Sum Query - Mutable | Medium | BIT as simpler alternative |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Fenwick = simpler segment tree for prefix sums ONLY           │
├──────────────────────────────────────────────────────────────────────┤
│  📌 i & (-i) = lowest set bit = the Fenwick navigation key        │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Update: walk UP (+= lowbit), Query: walk DOWN (-= lowbit)     │
├──────────────────────────────────────────────────────────────────────┤
│  📌 "Count smaller to right" → BIT on values + reverse traversal  │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Advanced Union-Find